# Evaluating LLM Agents: From Vibes to Metrics

**Workshop**: Building Trustworthy LLM Agents — PyConf Hyderabad 2026

**Section 2**: Evals & Evaluation Frameworks

---

> This notebook covers the **theory and concepts** behind evaluating LLM agents.  
> All practical eval code lives in the `evals/` test files and will be run live via `pytest`.

---

## Our Project: Support Swarm

We're building a **multi-agent customer support system** using LangGraph. It handles real tasks — order lookups, refunds, policy questions, and escalations — using multiple specialist agents.

```
User Query
    │
    ▼
┌──────────┐
│  Router  │  ← Classifies intent via structured output
└──────────┘
    │
    ├─ general ────────► ShopAssist        (lookup_order, process_refund, send_email)
    ├─ order_support ──► ShopAssist
    ├─ policy_inquiry ─► PolicyAdvisor     (search_knowledge_base)
    └─ escalation ─────► EscalationAgent   (search_knowledge_base, send_email)
```

The agents are backed by a **PostgreSQL + pgvector** database with customer records, orders, and a knowledge base of support articles. Each agent has a YAML-defined persona, rules, and allowed tools.

### Why does this project need evals?

This system has all the ingredients for silent failures:

- **Routing decisions** — if the router misclassifies intent, the wrong agent handles the request
- **Tool sequencing** — the agent must call `lookup_order` *before* `process_refund`, never the other way around
- **Policy enforcement** — refunds above $150 must be declined, not auto-approved
- **Grounded responses** — answers must come from retrieved data, not hallucinated
- **Security** — order data can contain injected instructions (prompt injection via tool output)

None of these failure modes are visible in the chat UI. The agent always responds confidently. **Evals are how we catch what our eyes miss.**

---

## Why Traditional Testing Fails for Agents

Traditional software testing relies on **deterministic behavior**: given input X, expect output Y. LLM agents break this in fundamental ways:

| Traditional Software | LLM Agent |
|---|---|
| Deterministic output | Stochastic — same input, different outputs |
| Binary pass/fail | Subjective quality — "good enough" vs "perfect" |
| Fixed control flow | Dynamic tool selection — agent *chooses* what to do |
| Input validation | Natural language — infinite input space |
| Unit testable functions | Multi-step reasoning chains |

### Example from our agent

Consider: *"I want a refund for my headphones"*

All of these are correct responses:
- "I'd be happy to help with your refund. Could you provide your order ID?"
- "Sure! Please share your order number so I can look that up."
- "Of course. What's the order ID for the headphones?"

**None would pass an exact-match test.** We need a fundamentally different approach — one that evaluates *quality* and *correctness* rather than exact string matching.

---

## The Evaluation Taxonomy

### What to Evaluate in an Agent System

Evaluation for agents happens at multiple levels — from low-level tool calls to end-to-end customer experience:

```
┌─────────────────────────────────┐
│  Level 4: End-to-End Quality    │  ← Is the customer satisfied?
├─────────────────────────────────┤
│  Level 3: Response Quality      │  ← Is the response faithful, relevant, hallucination-free?
├─────────────────────────────────┤
│  Level 2: Tool Usage            │  ← Did the agent call the right tools with right args?
├─────────────────────────────────┤
│  Level 1: Routing               │  ← Did the router pick the correct specialist agent?
└─────────────────────────────────┘
```

### When to Evaluate

| Stage | Method | Speed | Coverage |
|-------|--------|-------|----------|
| **Development** | Golden dataset + offline evals | Fast | Fixed scenarios |
| **CI/CD** | Automated test suite (`pytest` / `deepeval test run`) | Minutes | Regression detection |
| **Production** | Online eval (sample + LLM-as-judge) | Async | Real traffic |
| **Review** | Human evaluation + labeling | Slow | Ground truth creation |

### LLM-as-Judge: The Key Concept

Since we can't use exact matching, we use **another LLM to evaluate the agent's output**. This is called **LLM-as-Judge**.

```
User Question ──┐
                 ├──► Judge LLM ──► Score (0.0 - 1.0)
Agent Response ──┤                  + Reasoning
Ground Truth ────┘
```

The judge receives the question, the agent's response, and optionally ground truth data, then produces a score with reasoning. Most evaluation frameworks are built around this pattern.

---

## Types of Eval Metrics

There's no single metric that tells you "the agent is good." Different aspects of an agent require different kinds of evaluation. Here's a breakdown of the major metric categories:

### Agent Metrics (Tool Use & Orchestration)

These measure whether the agent took the **right actions** — not just whether it said the right thing.

| Metric | What It Measures |
|--------|------------------|
| **Tool Correctness** | Did the agent call the right tools in the right order with the right arguments? |
| **Task Completion** | Did the agent actually accomplish the user's goal end-to-end? |
| **Policy Adherence** | Did the agent follow business rules (e.g. refund caps, required steps)? |

```
Expected: lookup_order(ORD-1001) → process_refund(ORD-1001, 49.99) → send_email(alice@...)
Actual:   process_refund(ORD-1001, 49.99) → send_email(alice@...)   ← FAIL: skipped lookup
```

### RAG Metrics (Retrieval-Augmented Generation)

These measure whether the agent's response is **grounded in retrieved data** rather than hallucinated.

| Metric | What It Measures |
|--------|------------------|
| **Faithfulness** | Are all claims in the response supported by the retrieved context? |
| **Hallucination** | Did the agent make up facts not present in any provided context? |
| **Context Precision** | Is the retrieved context actually relevant to the question? |
| **Context Recall** | Does the retrieved context contain all the info needed to answer? |
| **Answer Relevancy** | Does the response actually address what the user asked? |

```
Retrieved:  "Items may be returned within 30 calendar days."
✓ Faithful: "You have 30 days to return items."
✗ Hallucinated: "You have 60 days to return items."
```

### Multi-Turn Metrics

Single-turn evals miss a critical dimension — can the agent maintain context across a conversation?

| Metric | What It Measures |
|--------|------------------|
| **Context Retention** | Does the agent remember earlier parts of the conversation? |
| **Disambiguation** | Can the agent resolve references like "that order" or "the first one"? |
| **Conversation Coherence** | Does the agent's behavior stay consistent across turns? |

### Safety & Security Metrics

These measure whether the agent can withstand adversarial inputs and stay within bounds.

| Metric | What It Measures |
|--------|------------------|
| **Prompt Injection Resilience** | Does the agent ignore injected instructions in tool outputs or user messages? |
| **Toxicity** | Does the agent generate harmful, offensive, or biased content? |
| **PII Leakage** | Does the agent expose sensitive information it shouldn't? |
| **Scope Adherence** | Does the agent refuse off-topic requests appropriately? |

### MCP (Model Context Protocol) Metrics

As agents connect to external tools via MCP, new evaluation dimensions emerge:

| Metric | What It Measures |
|--------|------------------|
| **Tool Selection Accuracy** | Does the agent pick the right MCP tool from a large registry? |
| **Parameter Extraction** | Are tool arguments correctly extracted from natural language? |
| **Error Recovery** | Does the agent handle tool failures or unexpected responses gracefully? |

### Custom / Domain-Specific Metrics (GEval)

Not everything maps to a built-in metric. **GEval** lets you define a custom rubric and use an LLM as judge.

| Metric | What It Measures |
|--------|------------------|
| **Empathy** | Does the response acknowledge the customer's frustration? |
| **Professionalism** | Is the tone appropriate for a support interaction? |
| **Completeness** | Did the response cover all parts of the user's question? |

```
Evaluation Criteria: Rate the response on empathy (1-5).
- 5: Acknowledges frustration, shows understanding
- 3: Neutral, functional response
- 1: Dismissive, robotic, or cold
```

---

## Evaluation Frameworks: A Comparison

There are several frameworks available for evaluating LLM applications. Here's how the major ones compare:

| Framework | Best For | Key Strengths | Limitations |
|-----------|----------|---------------|-------------|
| **DeepEval** | Agent evals, CI/CD integration | Tool correctness, GEval custom metrics, pytest-native, built-in dashboard | Smaller community than some alternatives |
| **RAGAS** | RAG pipeline evaluation | Claim-level faithfulness decomposition, comprehensive RAG metrics | Focused primarily on RAG, less on agent orchestration |
| **LangSmith** | LangChain ecosystem | Tight LangChain integration, tracing + evals in one platform | Vendor lock-in to LangChain ecosystem |
| **Promptfoo** | Prompt engineering & red-teaming | Config-driven, great for prompt comparison, security testing | Less focus on agent-level orchestration |
| **Braintrust** | Production monitoring | Logging + evals combined, real-time scoring | Primarily a hosted platform |
| **Arize Phoenix** | Observability + evals | Tracing and eval in one tool, open-source | More observability-focused than eval-focused |

### How to Choose?

Ask yourself these questions:

1. **What are you evaluating?**
   - RAG pipeline → **RAGAS** excels at faithfulness and context quality
   - Agent tool use → **DeepEval** has `ToolCorrectnessMetric` built-in
   - Prompt quality → **Promptfoo** is great for A/B testing prompts

2. **Where do you need evals?**
   - CI/CD pipeline → **DeepEval** (pytest integration, `deepeval test run`)
   - Production monitoring → **Braintrust** or **Arize Phoenix**
   - Development iteration → **LangSmith** or **RAGAS**

3. **How custom are your metrics?**
   - Standard metrics (faithfulness, relevancy) → Most frameworks cover these
   - Custom rubrics (empathy, domain compliance) → **DeepEval's GEval** or **RAGAS** custom metrics

### Our Choice: DeepEval (primary) + RAGAS (RAG-specific)

For this workshop, we use **DeepEval** as our primary framework because:
- It integrates natively with **pytest** — our tests are just test functions
- It has **`ToolCorrectnessMetric`** — purpose-built for evaluating agent tool calls
- It supports **GEval** — letting us define custom rubrics for empathy, scope, and injection resilience
- It provides a **dashboard** via `deepeval test run` for tracking scores over time

We complement it with **RAGAS** for deeper **faithfulness analysis** on our RAG-based agents (PolicyAdvisor, ShopAssist), since RAGAS does claim-level decomposition that gives more granular insight into *what* went wrong.

| Aspect | DeepEval | RAGAS |
|--------|----------|-------|
| **Integration** | pytest via `assert_test()` | `evaluate()` returns DataFrame |
| **Granularity** | Single score per metric | Claim-level decomposition |
| **Best for** | CI/CD gating, agent evals | Detailed RAG analysis and debugging |
| **In our project** | `test_tool_correctness.py`, `test_faithfulness.py`, `test_custom_quality.py`, `test_security.py` | `test_ragas.py` |

---

## DeepEval: A Quick Introduction

[DeepEval](https://github.com/confident-ai/deepeval) is an open-source evaluation framework for LLM applications. It's designed to work like a **unit testing framework** — you write test cases, define metrics, set thresholds, and run them with `pytest`.

### Core Concepts

**1. `LLMTestCase`** — The unit of evaluation

Each test case wraps the input, the agent's output, and any supporting context:

```python
from deepeval.test_case import LLMTestCase

test_case = LLMTestCase(
    input="I want a refund for ORD-1001",
    actual_output="I've processed your refund of $49.99.",
    expected_output="Refund processed for ORD-1001",
    retrieval_context=["Order ORD-1001: Wireless Headphones, $49.99, delivered"],
    expected_tools=["lookup_order", "process_refund"],
    actual_tools=["lookup_order", "process_refund"],
)
```

**2. Metrics** — What to measure

DeepEval provides built-in metrics that use LLM-as-Judge under the hood:

| Metric | Purpose |
|--------|---------|
| `ToolCorrectnessMetric` | Compares expected vs actual tool calls |
| `FaithfulnessMetric` | Checks if response is grounded in retrieval context |
| `HallucinationMetric` | Detects fabricated information |
| `AnswerRelevancyMetric` | Checks if response addresses the question |
| `GEval` | Custom rubric — you define the criteria |

**3. `assert_test()`** — The assertion

```python
from deepeval import assert_test

assert_test(test_case, metrics=[ToolCorrectnessMetric()])
```

If the metric score falls below the threshold, the test fails — just like a regular assertion.

**4. Running evals**

```bash
# Via pytest (what we use)
pytest evals/test_tool_correctness.py -v

# Via deepeval CLI (adds dashboard + reporting)
deepeval test run evals/ --report
```

### GEval: Build Your Own Metric

GEval is DeepEval's most flexible feature — it lets you define **any evaluation criterion** as natural language:

```python
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

empathy_metric = GEval(
    name="Empathy & Professionalism",
    criteria="Evaluate whether the response acknowledges the customer's "
             "situation and responds with appropriate empathy.",
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    threshold=0.6,
)
```

The LLM judge reads your criteria and scores the response accordingly. This is how we evaluate things like **empathy**, **scope adherence**, and **injection resilience** — dimensions that no pre-built metric covers.

---

## Let's See It in Action

Now that we understand the theory — **why** we need evals, **what** types of metrics exist, and **which** frameworks to use — let's run them against our live agent.

### What we'll do in the practical demo

We'll run `pytest` against the running LangGraph server to evaluate our agent across multiple dimensions:

| Eval | Test File | What It Catches |
|------|-----------|----------------|
| Policy Boundaries | `test_tool_correctness.py` | Agent auto-approving refunds above the $150 cap |
| Faithfulness | `test_faithfulness.py` | Agent making claims not supported by retrieved data |
| RAGAS Faithfulness | `test_ragas.py` | Claim-level grounding analysis for RAG responses |
| Routing Accuracy | `test_routing.py` | Router misclassifying user intent |
| Multi-Turn Context | `test_multiturn.py` | Agent losing context across conversation turns |
| Custom Quality | `test_custom_quality.py` | Empathy, professionalism, scope adherence via GEval |
| Security | `test_security.py` | Indirect prompt injection via poisoned tool output |
| Escalation | `test_escalation.py` | Priority classification and team routing |

### The pattern for every demo

1. **Chat with the agent** in the UI — it looks fine
2. **Run the eval** — it reveals silent failures
3. **Fix the agent** — update the YAML config
4. **Re-run the eval** — scores improve

> **The gap between "looks fine in chat" and "passes evals" is exactly why we need automated evaluation.**